In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
df = pd.read_csv(r"/content/Restaurant_Reviews.tsv", delimiter='\t')
df.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [5]:
ps = PorterStemmer()
all_stopwords = stopwords.words('english')

if 'not' in all_stopwords:
    all_stopwords.remove('not')

corpus = []

for review in df['Review']:
    review = re.sub('[^a-zA-Z]', ' ', str(review))
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if word not in all_stopwords]
    review = ' '.join(review)
    corpus.append(review)

df['Clean_Review'] = corpus
df.head()

,Review,Liked,Clean_Review
0,Wow... Loved this place.,1,wow love place
1,Crust is not good.,0,crust not good
2,Not tasty and the texture was just nasty.,0,not tasti textur nasti
3,Stopped by during the late May bank holiday of...,1,stop late may bank holiday rick steve recommen...
4,The selection on the menu was great and so wer...,1,select menu great price


In [6]:
X = df['Clean_Review']
y = df['Liked']

vectorizer = TfidfVectorizer(max_features=1500)
X = vectorizer.fit_transform(X)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [8]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.785

Classification Report:

              precision    recall  f1-score   support

           0       0.73      0.88      0.80        97
           1       0.86      0.70      0.77       103

    accuracy                           0.79       200
   macro avg       0.79      0.79      0.78       200
weighted avg       0.80      0.79      0.78       200



In [9]:
negative_reviews = df[df['Liked'] == 0]['Clean_Review']

words = " ".join(negative_reviews).split()
word_freq = Counter(words)

print("Top 10 common words in negative reviews:")
for word, count in word_freq.most_common(10):
    print(word, ":", count)

Top 10 common words in negative reviews:
not : 98
food : 67
place : 52
servic : 40
go : 39
back : 38
like : 31
time : 29
wait : 26
disappoint : 23


In [10]:
sample = ["food was cold and service was very slow"]

sample_clean = []
for review in sample:
    review = re.sub('[^a-zA-Z]', ' ', review)
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if word not in all_stopwords]
    review = ' '.join(review)
    sample_clean.append(review)

sample_vector = vectorizer.transform(sample_clean)
prediction = model.predict(sample_vector)

if prediction[0] == 1:
    print("Sentiment: Positive")
else:
    print("Sentiment: Negative")

Sentiment: Negative
